<a href="https://colab.research.google.com/github/daviseemann/turbofan-rul-prediction-cmapss/blob/production/notebooks/autoencoder-0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive

drive.mount(
    "/content/drive/",
)
import os

os.chdir(
    "/content/drive/MyDrive/Data science studies/Aprendizado-de-maquina-UFSC/final-project/data"
)

# Importando os dados


In [ ]:
# Caminhos dos arquivos
train_path = "./6.turbofan rul/train_FD001.txt"
test_path = "./6.turbofan rul/test_FD001.txt"
rul_path = "./6.turbofan rul/RUL_FD001.txt"

# Nomes das colunas (de acordo com a documentação original do C-MAPSS)
column_names = (
    ["engine_id", "cycle"]
    + [f"op{i}" for i in range(1, 4)]
    + [f"s{i}" for i in range(1, 22)]
)

# Importando os arquivos (espaço em branco como delimitador)
df_train = pd.read_csv(train_path, sep="\s+", header=None, names=column_names)
df_test = pd.read_csv(test_path, sep="\s+", header=None, names=column_names)
df_rul = pd.read_csv(rul_path, sep="\s+", header=None, names=["RUL"])

O dataset C-MAPSS possui os seguintes campos:

- `engine_id`: identificador do motor
- `cycle`: número do ciclo de operação
- `op1` a `op3`: variáveis de configuração operacional
- `s1` a `s21`: leituras sensoriais

Neste notebook, utilizaremos apenas os dados do conjunto FD001 (falha única, condições constantes).


In [ ]:
df_train.head()

# Tratamento dos dados


## Seleção das features

Com base no artigo, foram selecionadas apenas as features consideradas mais relevantes por estudos anteriores. Esses estudos identificaram sensores com maior variância e correlação com a RUL, descartando sensores redundantes ou pouco informativos.

Os sensores selecionados são:

{2, 3, 4, 7, 8, 9, 11, 12, 13, 14, 15, 17, 20, 21}

Essa seleção ajuda a reduzir o ruído e melhorar a performance do modelo.


In [ ]:
def get_features(df):
    """Filtra o DataFrame mantendo apenas as colunas consideradas relevantes conforme o artigo base."""
    selected_indices = [2, 3, 4, 7, 8, 9, 11, 12, 13, 14, 15, 17, 20, 21]
    selected_sensors = [f"s{idx}" for idx in selected_indices]
    # print(selected_sensors)
    required_cols = ["engine_id", "cycle"] + selected_sensors
    df_filtered = df[required_cols].copy()
    return df_filtered

## Escalar os dados usando o MinMaxScalar

Importante se atentar aqui que o scaler foi desenvolvido para fazer o tratamento dos dados segmentados pelo motor, evitando assim um data leakege.


In [ ]:
from sklearn.preprocessing import MinMaxScaler


def scale_per_engine(
    df,
    feature_cols=[
        "s2",
        "s3",
        "s4",
        "s7",
        "s8",
        "s9",
        "s11",
        "s12",
        "s13",
        "s14",
        "s15",
        "s17",
        "s20",
        "s21",
    ],
):
    """Aplica MinMaxScaler para cada motor individualmente."""
    df_scaled = df.copy()

    # Converter as colunas de features para float antes de escalar
    df_scaled[feature_cols] = df_scaled[feature_cols].astype(float)

    for eid in df_scaled["engine_id"].unique():  # Usar df_scaled aqui
        idx = df_scaled["engine_id"] == eid  # Usar df_scaled aqui
        scaler = MinMaxScaler()
        # A atribuição agora é entre tipos de dados compatíveis (float)
        df_scaled.loc[idx, feature_cols] = scaler.fit_transform(
            df_scaled.loc[idx, feature_cols]
        )

    return df_scaled

## Fazendo o calculo do RUL

Isso foi definido baseado no paper de referencia da MLP que usa um conceito mostrado no paper oficial do dataset. Onde estima-se um RUL máximo sobre o motor onde está considerando o comportamento "normal".


In [ ]:
# Cálculo da vida útil
def compute_rul(df, max_rul=125):
    """Calcula o RUL com limite superior (clipping)."""
    df = df.copy()
    max_cycle = df.groupby("engine_id")["cycle"].transform("max")
    df["RUL"] = (max_cycle - df["cycle"]).clip(upper=max_rul)
    return df

## Fazendo a divisão dos dados em janelas

Uma MLP não consegue entender dados sequenciais de forma precisa, então para resolver esse problema foi definida a arquitetura de janelas. Desta forma, n_w amostras são empacotadas em um vetor e atribuida a label do rul de seu ultimo ciclo. Isso ajuda a MLP a compreender os dados de maneira mais precisa fazendo uma estimativa mais congruente com a natureza do problema.


In [ ]:
def generate_windows(df, sequence_length=30, features=None):
    """Gera janelas temporais com flattened input e target RUL."""
    X, y = [], []
    for eid in df["engine_id"].unique():
        engine_df = df[df["engine_id"] == eid].sort_values("cycle")
        sensor_data = engine_df[features].values
        rul_data = engine_df["RUL"].values

        for i in range(len(engine_df) - sequence_length + 1):
            window = sensor_data[i : i + sequence_length].flatten()
            target = rul_data[i + sequence_length - 1]
            X.append(window)
            y.append(target)

    return np.array(X), np.array(y)

In [ ]:
def plot_flattened_window(
    X_array, sample_index, sequence_length, num_features, feature_names=None
):
    """
    Plota os dados de uma janela achatada do X_train.
    """
    if sample_index >= len(X_array):
        print(
            f"Índice da amostra ({sample_index}) fora dos limites de X_array (tamanho {len(X_array)})."
        )
        return

    # Seleciona a janela achatada
    flattened_window = X_array[sample_index]

    # Desachata a janela de volta para (sequence_length, num_features)
    try:
        window_data = flattened_window.reshape(sequence_length, num_features)
    except ValueError:
        print(
            f"Erro ao desachatar: O tamanho da janela ({flattened_window.shape[0]}) não corresponde a sequence_length * num_features ({sequence_length * num_features})."
        )
        return

    fig, axes = plt.subplots(
        num_features, 1, figsize=(10, 2 * num_features), sharex=True
    )

    if num_features == 1:
        axes = [axes]

    for i in range(num_features):
        ax = axes[i]
        feature_name = (
            feature_names[i]
            if feature_names and len(feature_names) > i
            else f"Feature {i+1}"
        )
        ax.plot(window_data[:, i], label=feature_name)
        ax.set_ylabel(feature_name)
        ax.grid(True)
        ax.legend(loc="upper right")

    axes[-1].set_xlabel("Passos na Janela (Ciclos)")
    fig.suptitle(
        f"Visualização da Janela Achatada (Amostra {sample_index})", fontsize=14
    )
    plt.tight_layout()
    plt.show()

## Divisão entre treino e teste


Importante se atentar em não ocorrer data leakage, assim serão dividios os dados de forma que nenhum motor apareça no conjunto de teste e validação. Evitando um possível data leakege atrapalhando nosso controle sobre o treino e interpretação dos resultados.


In [ ]:
from sklearn.model_selection import train_test_split


def split_by_engine(df, test_size=0.2, random_state=42):
    """Divide o dataset com base nos engine_ids (sem vazamento temporal)."""
    engine_ids = df["engine_id"].unique()
    train_ids, val_ids = train_test_split(
        engine_ids, test_size=test_size, random_state=random_state
    )

    df_train = df[df["engine_id"].isin(train_ids)].reset_index(drop=True)
    df_val = df[df["engine_id"].isin(val_ids)].reset_index(drop=True)

    return df_train, df_val

## Pipeline de tratamento dos dados


In [ ]:
def prepare_data_for_mlp(
    df, sequence_length=30, max_rul=125, test_size=0.2, random_state=42
):
    df_filtered = get_features(df)
    df_rul = compute_rul(df_filtered, max_rul=max_rul)
    df_scaled = scale_per_engine(df_rul)
    feature_cols = [col for col in df_scaled.columns if col.startswith("s")]
    df_train, df_val = split_by_engine(
        df_scaled, test_size=test_size, random_state=random_state
    )
    X_train, y_train = generate_windows(
        df_train, sequence_length=sequence_length, features=feature_cols
    )
    X_val, y_val = generate_windows(
        df_val, sequence_length=sequence_length, features=feature_cols
    )
    return (
        X_train.astype("float32"),
        y_train.astype("float32"),
        X_val.astype("float32"),
        y_val.astype("float32"),
    )

In [ ]:
# Processando os dados para a MLP
X_train, y_train, X_val, y_val = prepare_data_for_mlp(
    df_train, sequence_length=30, max_rul=125, test_size=0.2, random_state=42
)

X_train.shape, y_train.shape, X_val.shape, y_val.shape

In [ ]:
feature_cols = [
    "s2",
    "s3",
    "s4",
    "s7",
    "s8",
    "s9",
    "s11",
    "s12",
    "s13",
    "s14",
    "s15",
    "s17",
    "s20",
    "s21",
]

# Chame a nova função
plot_flattened_window(
    X_train,
    sample_index=1,
    sequence_length=30,
    num_features=len(feature_cols),
    feature_names=feature_cols,
)

# Construção do modelo MLP


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import tensorflow.keras.backend as K

In [ ]:
train_dataset = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train))
    .batch(64)
    .prefetch(tf.data.AUTOTUNE)
)
val_dataset = (
    tf.data.Dataset.from_tensor_slices((X_val, y_val))
    .batch(64)
    .prefetch(tf.data.AUTOTUNE)
)

In [ ]:
def build_mlp(input_dim, learning_rate=0.001):
    """Cria uma MLP simples para regressão de RUL."""
    model = Sequential(
        [
            Input(shape=(input_dim,)),
            Dense(100, activation="relu"),
            Dense(50, activation="relu"),
            Dense(1),  # saída: valor contínuo do RUL
        ]
    )
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="mse",
        metrics=[rmse_metric, nasa_score_metric],
    )
    return model

In [ ]:
def rmse_metric(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))


def nasa_score_metric(y_true, y_pred):
    diff = y_pred - y_true
    score = K.switch(
        K.greater(diff, 0), K.exp(diff / 13.0) - 1, K.exp(-diff / 10.0) - 1
    )
    return K.mean(score)

In [ ]:
def train_mlp(model, X_train, y_train, X_val, y_val, epochs=100, batch_size=64):
    early_stop = EarlyStopping(
        monitor="val_loss", patience=10, restore_best_weights=True
    )
    checkpoint = ModelCheckpoint(
        "best_mlp_model.h5", save_best_only=True, monitor="val_loss"
    )

    history = model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=100,
        callbacks=[early_stop, checkpoint],
        verbose=1,
    )
    return history

In [ ]:
import matplotlib.pyplot as plt


def plot_learning_curves(history):
    plt.figure(figsize=(18, 4))

    # Plot Loss (MSE)
    plt.subplot(1, 3, 1)
    plt.plot(history.history["loss"], label="Treino")
    plt.plot(history.history["val_loss"], label="Validação")
    plt.title("Loss (MSE)")
    plt.xlabel("Epochs")
    plt.ylabel("MSE")
    plt.legend()
    plt.grid(True)

    # Plot RMSE
    if "rmse_metric" in history.history:
        plt.subplot(1, 3, 2)
        plt.plot(history.history["rmse_metric"], label="Treino")
        plt.plot(history.history["val_rmse_metric"], label="Validação")
        plt.title("RMSE")
        plt.xlabel("Epochs")
        plt.ylabel("RMSE")
        plt.legend()
        plt.grid(True)

    # Plot NASA Score
    if "nasa_score_metric" in history.history:
        plt.subplot(1, 3, 3)
        plt.plot(history.history["nasa_score_metric"], label="Treino")
        plt.plot(history.history["val_nasa_score_metric"], label="Validação")
        plt.title("NASA Score")
        plt.xlabel("Epochs")
        plt.ylabel("NASA Score")
        plt.legend()
        plt.grid(True)

    plt.tight_layout()
    plt.show()

In [ ]:
X_train, y_train, X_val, y_val = prepare_data_for_mlp(df_train)

model = build_mlp(X_train.shape[1])
history = train_mlp(model, X_train, y_train, X_val, y_val)
plot_learning_curves(history)